In [10]:
import requests
import pandas as pd
from pathlib import Path


In [5]:
#Lagoa do Ouro - PE
LATITUDE = -9.1269
LONGITUDE = -36.46

START_DATE = "2024-01-01"
END_DATE = "2025-12-01"

ARCH_URL = "https://archive-api.open-meteo.com/v1/archive" 

In [7]:
def extract_weather_data(lat, lon, st_dt, end_dt, url):
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": st_dt,
        "end_date": end_dt,
        "hourly": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m",
        "timezone": "America/Sao_Paulo" 
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        raise Exception(f"API request failed: {response.status_code} - {response.text}")
    
    return response.json()

raw_json = extract_weather_data(LATITUDE, LONGITUDE, START_DATE, END_DATE, ARCH_URL)

In [9]:
def transform_data(json_data):
    hourly_data = json_data["hourly"]

    df = pd.DataFrame(hourly_data)

    df["time"] = pd.to_datetime(df["time"])

    column_map = {
        "temperature_2m": "temperature",
        "relative_humidity_2m": "humidity_pc",
        "precipitation": "precip_mm",
        "wind_speed_10m": "wind_speed_kmh"
    }

    df = df.rename(columns=column_map)

    df = df.dropna()

    return df

df = transform_data(raw_json)
print(df)

                     time  temperature  humidity_pc  precip_mm  wind_speed_kmh
0     2024-01-01 00:00:00         22.5           86        0.0             8.7
1     2024-01-01 01:00:00         22.0           89        0.0             7.7
2     2024-01-01 02:00:00         21.2           93        0.0             7.8
3     2024-01-01 03:00:00         20.7           96        0.0             6.5
4     2024-01-01 04:00:00         20.5           96        0.0             6.8
...                   ...          ...          ...        ...             ...
16819 2025-12-01 19:00:00         22.7           70        0.0            18.5
16820 2025-12-01 20:00:00         21.5           78        0.0            17.8
16821 2025-12-01 21:00:00         20.8           87        0.0            17.7
16822 2025-12-01 22:00:00         20.4           90        0.0            16.5
16823 2025-12-01 23:00:00         19.9           93        0.0            15.0

[16824 rows x 5 columns]


In [16]:
def load_data(df, path_str):
    path = Path(path_str)

    path.parent.mkdir(parents=True, exist_ok=True)

    df.to_csv(path, index=False)

    print(f"Data saved to: {path}\nShape: {df.shape}")

path = "data/weather.csv"

load_data(df, path)

Data saved to: data\weather.csv
Shape: (16824, 5)
